# Selection editor

<video controls src="./assets/mobile_restraints.webm">

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: selection editor")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

In [3]:
utilities.use_recording_commands()
utilities.use_interaction_modes()
utilities.selections.update_selection("root", renderer="ball and stick")

## Interaction & commands

Mode that adds/removes a restraint whenever an interaction begins:

In [4]:
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode

class ToggleMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        if "is_restraint" in interaction.properties:
            return

        selection = get_current_selection()
        particles = set(selection.get("selected", {}).get("particle_ids", []))

        if particles.intersection(interaction.particles):
            particles.difference_update(interaction.particles)
        else:
            particles.update(interaction.particles)

        utilities.notify_all(f"{selection}")
        utilities.selections.update_selection(selection["id"].removeprefix("selection."), particle_ids=list(map(int, particles)), renderer=selection.get("properties", {}).get("nanover.rendering.renderer", "liquorice"))

utilities.add_interaction_mode(ToggleMode, "toggle", icon="🫧")

In [5]:
RENDERERS = ["liquorice", "ball and stick", "cartoon"]
CURRENT_SELECTION_INDEX = 0

def get_current_selection():
    selections = list(utilities.selections.all_prefixed())
    selections.sort()
    d = {key:value for key, value in utilities.selections.all_prefixed_items()}
    return d[selections[CURRENT_SELECTION_INDEX]]

def cycle_renderer():
    selection = get_current_selection()

    try:
        current = RENDERERS.index(selection.get("properties", {}).get("nanover.rendering.renderer", None))
        next = (current+1) % len(RENDERERS)
    except ValueError:
        next = 0

    utilities.selections.update_selection(selection["id"].removeprefix("selection."), particle_ids=selection["selected"]["particle_ids"], renderer=RENDERERS[next])
    refresh_panels()


def prev_selection():
    global CURRENT_SELECTION_INDEX

    selections = utilities.selections.all_prefixed()
    CURRENT_SELECTION_INDEX = (CURRENT_SELECTION_INDEX - 1) % len(selections)

    refresh_panels()

def next_selection():
    global CURRENT_SELECTION_INDEX

    selections = utilities.selections.all_prefixed()
    CURRENT_SELECTION_INDEX = (CURRENT_SELECTION_INDEX + 1) % len(selections)

    refresh_panels()

def create_selection():
    refresh_panels()

def delete_selection():
    refresh_panels()

imd_runner.app_server.register_command("selections/cycle-renderer", cycle_renderer)
imd_runner.app_server.register_command("selections/prev", prev_selection)
imd_runner.app_server.register_command("selections/next", next_selection)
imd_runner.app_server.register_command("selections/create", create_selection)
imd_runner.app_server.register_command("selections/delete", delete_selection)

In [6]:
def refresh_panels():
    current = get_current_selection()

    utilities.panels.update_panel(
        "test",
        utilities.panels.header(label=f"Selection {current["id"]}"),
        utilities.panels.button(label=f"Renderer: {current.get("properties", {}).get("nanover.rendering.renderer", None)}", command="selections/cycle-renderer"),
        utilities.panels.button(label="next", command="selections/next"),
        utilities.panels.button(label="prev", command="selections/prev"),
        utilities.panels.button(label="create", command="selections/create"),
        utilities.panels.button(label="delete", command="selections/delete"),
        label="Edit selections",
    )

refresh_panels()

In [7]:
utilities.show_logging()

Output()